# Pair Distributions

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

# Corrected TNG300 comparisons

Restart the kernel and run from the top. Set `LG_PROJECT` to your project directory on Roihu before running the setup cell. This notebook reads the corrected selection and existing feature caches; it does not require newly trained NPE checkpoints.

Inputs are verified against the v3 manifests. The corrected TNG sample has 1,530 pairs: 1,253 same-host and 277 different-host. Figures are written to `pickles_v3/figures/Pair_Distributions`. Selection and feature caches are read only. Both simulations' radial pair velocities are **peculiar**; these comparison plots do not add a Hubble term.

Pair distributions count one row per pair. Galaxy-feature distributions explicitly distinguish unique final-pair members from pair-member entries (a galaxy may appear repeatedly). KS/energy/Wasserstein distances are descriptive; shared galaxies and hosts invalidate naive independent-row significance claims. Historical interpretation notes below require reassessment after rerunning.

Raw SOAP metadata, raw TNG catalogue checks, and large parent-catalogue diagnostics are optional flags in the setup cell and are off by default. Existing originals and outputs are preserved in the dated comparison-update backup.


In [ ]:
import os, sys
from pathlib import Path
PROJECT = Path(os.environ.get("LG_PROJECT", r"<research-project>"))
sys.path.insert(0, str(PROJECT / "analogues"))
from review_fixes import (RUN_VERSION, MODEL_DESCRIPTION, H0_BY_SIM, configure_run,
    feature_cache_dir, lg_observation, lg_to_peculiar, draw_noisy_feature,
    sample_lg_point, sample_lg_marginalized, calibration_summary, tng_volume_mpc3)
LEGACY_RESULTS_DIR = PROJECT / "pickles_v2"
FEATURE_CACHE_DIR = feature_cache_dir(PROJECT)


from comparison_utils import (load_comparison_inputs, finite_lookup,
    normalized_wasserstein, group_blocks, resample_blocks)
from splits import pair_groups, group_split
RESULTS_DIR, PLOT_DIR, tng_dir, TNG_DATA, TNG_MANIFEST = load_comparison_inputs(
    PROJECT, 'Pair_Distributions')
RUN_SOAP_METADATA = False  # Optional remote metadata lookup; requires hdfstream.
RUN_TNG_CATALOGUE_CHECKS = False  # Optional raw catalogue; set TNG_BASE when enabled.
RUN_PARENT_CATALOGUE_CHECKS = False  # Optional large FLAMINGO parent-cache diagnostic.

### Introduction

This notebook compares the local group analogues obtained after applying the enitire pipeline in flamingo_lg_pipeline.py for FLAMINGO and TNG300. The goal of this notebook is to verify various hypothesis and identify differences and similarities that may be useful in the inference section.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

#### Filters applied pairs

Filters used are:
- n_stars>=100 (Quality filter)
- Stellar mass 2e10 - 5e11
- seperation 500-1000 kpc
- No third subhalo with mass >= 1.5 * max(M_star_i, M_star_j) in the same host
- No intruder within 2000 kpc of pair CoM with mass > 0.5 × (M_i + M_j)
- vr: −400 to 0 km/s
- vt: 0 to 300 km/s
- v_total <= 500 km/s

All filters are consistent with TNG300.


In [ ]:
# # All filters are in a json file and pipeline is run with those filters as mentioned above.
# soap_file = open_default_soap()
# cfg = load_selection_config(r"<research-project>\analogues\flamingo_selection_config.json")
# fields, sample, pairs = run_flamingo_pipeline_from_config(
#     config=cfg,
#     soap_file=soap_file,
#     verbose=True,
# )
# print(f"Final FLAMINGO pairs: {len(pairs['i']):,}")

In [ ]:
import numpy as np

tng_dir = PROJECT / "analogues/tng_for_comparison_v3"

tng_same_host = TNG_DATA["have_same_host"]
tng_stellar_mass = TNG_DATA["stellar_mass"]
tng_sep          = TNG_DATA["separation"]
tng_vr           = TNG_DATA["vr"]
tng_vt           = TNG_DATA["vt"]
tng_log_m200c    = TNG_DATA["log_m200c"]
tng_log_mass_ratio = TNG_DATA["log_mass_ratio"]

print(f"TNG pairs loaded: {len(tng_sep):,}")
tng_pairs_i, tng_pairs_j = TNG_DATA["pairs_i"], TNG_DATA["pairs_j"]
tng_member_idx = np.unique(np.concatenate([tng_pairs_i, tng_pairs_j]))
tng_stellar_mass_members = tng_stellar_mass[tng_member_idx]
tng_groups = pair_groups(pd.DataFrame({
    "host_i": TNG_DATA["grnr"][tng_pairs_i],
    "host_j": TNG_DATA["grnr"][tng_pairs_j],
}))

In [ ]:
# Read the cached FLAMINGO selection; do not rebuild or overwrite it.
import pickle
cache_dir = PROJECT / "analogues/cache/run_001_mstar2e10"
with open(cache_dir / "flamingo_pairs.pkl", "rb") as f:
    pairs = pickle.load(f)
with open(cache_dir / "flamingo_sample.pkl", "rb") as f:
    sample = pickle.load(f)
pairs["have_same_host"] = pairs["have_same_host"].astype(bool)
flam_member_idx = np.unique(np.concatenate([pairs["i"], pairs["j"]]))
flam_mstar = sample["stellar_mass"][flam_member_idx]
print(f"FLAMINGO: {len(pairs['i']):,} pairs, {len(flam_member_idx):,} unique pair members.")
print(f"TNG300: {len(tng_sep):,} pairs, {len(tng_member_idx):,} unique pair members.")

> Historical interpretation from before the corrected comparison rerun. Reassess quantitative and causal claims using the new outputs.

The two simulations show quite different stellar mass functions within the MW-mass window. FLAMINGO produces a broader distribution skewed toward higher stellar masses. This reflects differences in stellar feedback calibration between IllustrisTNG and FLAMINGO and motivates simulation-independent inference rather than direct comparison of pair counts.


In [ ]:
# Unique final-pair member masses, not a volume-normalised stellar mass function.

fig, ax = plt.subplots(figsize=(8, 5))
bins = np.logspace(10, 12, 40)

ax.hist(flam_mstar,       bins=bins, density=True, alpha=0.6, color="#E07B3F", label=f"FLAMINGO (N={len(flam_mstar):,})")
ax.hist(tng_stellar_mass_members, bins=bins, density=True, alpha=0.6, color="#4C72B0", label=f"TNG300 (N={len(tng_stellar_mass_members):,})")

ax.axvline(2e10, color="gray", linestyle="--", linewidth=1, label="current floor 2e10")
ax.set_xscale("log")
ax.set_xlabel("Stellar Mass [M☉]")
ax.set_ylabel("Density")
ax.set_title("Stellar mass — unique final-pair members")
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "smf_comparison.pdf", dpi=150, bbox_inches="tight")
plt.show()

Objects within the mass range have no objects below the threshold of n_stars>=100. This makes the n_stars filter redundant. The stellar mass function in range therefore is a result of difference in galaxy formation physics rather than resolution incompleteness.


In [ ]:
if RUN_PARENT_CATALOGUE_CHECKS:
    with open(cache_dir / "flamingo_fields.pkl", "rb") as f:
        fields = pickle.load(f)
    mstar = fields["stellar_mass"]
    nstar = fields["n_stars"]
    
    low_mass_all = (mstar >= 2e10) & (mstar <= 5e10)  # no quality flag
    low_mass_good = low_mass_all & fields["quality_flag"]  # with n_stars>=100
    
    print(f"Total objects in 2e10-5e10:              {low_mass_all.sum():,}")
    print(f"After n_stars>=100:                      {low_mass_good.sum():,}")
    print(f"Removed by quality cut:                  {(low_mass_all & ~fields['quality_flag']).sum():,}")
    
    print(f"\nn_stars in 2e10-5e10 BEFORE quality cut:")
    print(f"  median:          {np.median(nstar[low_mass_all]):.0f}")
    print(f"  10th percentile: {np.percentile(nstar[low_mass_all], 10):.0f}")
    print(f"  25th percentile: {np.percentile(nstar[low_mass_all], 25):.0f}")
    
    print(f"\nn_stars in 2e10-5e10 AFTER quality cut:")
    print(f"  median:          {np.median(nstar[low_mass_good]):.0f}")
    print(f"  10th percentile: {np.percentile(nstar[low_mass_good], 10):.0f}")
    print(f"  25th percentile: {np.percentile(nstar[low_mass_good], 25):.0f}")
else:
    print('Optional large FLAMINGO parent-catalogue quality diagnostic skipped.')

### Plots
Now we look at some distributions of quantities of both TNG300 and FLAMINGO to check key visual differences between them.


In [ ]:
FLAMINGO_COLOR = "#E07B3F"
TNG_COLOR      = "#4C72B0"

bins_sep   = np.linspace(500,  1000, 21)
bins_vr    = np.linspace(-400, 0,   21)
bins_vt    = np.linspace(0,    300, 21)
mass_min = min(np.min(pairs["log_tot_virial_mass"]), np.min(tng_log_m200c))
mass_max = max(np.max(pairs["log_tot_virial_mass"]), np.max(tng_log_m200c))
bins_m200c = np.linspace(mass_min - 1e-6, mass_max + 1e-6, 21)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle("FLAMINGO L1_m8 vs TNG300-1 — Local Group Analogues", fontsize=14)

panel_specs = [
    (axes[0, 0], pairs["separation_kpc"],      tng_sep,       bins_sep,   "Pair Separation [kpc]",             "Pair Separation"),
    (axes[0, 1], pairs["vr"],                  tng_vr,        bins_vr,    r"Peculiar radial velocity $v_r$ [km/s]",     "Radial Velocity"),
    (axes[1, 0], pairs["vt"],                  tng_vt,        bins_vt,    r"Tangential Velocity $v_t$ [km/s]", "Tangential Velocity"),
    (axes[1, 1], pairs["log_tot_virial_mass"],  tng_log_m200c, bins_m200c, r"$\log_{10}(M_{200c}/M_\odot)$",   "Host Halo Mass"),
]

for ax, flam_arr, tng_arr, bins, xlabel, title in panel_specs:
    vf = np.isfinite(flam_arr)
    vt = np.isfinite(tng_arr)
    ax.hist(flam_arr[vf], bins=bins, density=True, alpha=0.6, color=FLAMINGO_COLOR, label=f"FLAMINGO (N={vf.sum():,})")
    ax.hist(tng_arr[vt],  bins=bins, density=True, alpha=0.6, color=TNG_COLOR,      label=f"TNG300-1 (N={vt.sum():,})")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(PLOT_DIR / "flamingo_vs_tng_4panel.pdf", dpi=150, bbox_inches="tight")
plt.show()

From the above plot we can see some difference between the TNG300 transverse velocities and the radial velocity. This may be because of the difference of same-host fractions within the simulations which we will invesitgate further.


In [ ]:
flam_same = pairs["have_same_host"]

print("Same-host fraction:")
print(f"  FLAMINGO: {flam_same.mean():.1%}  ({flam_same.sum():,} / {len(flam_same):,})")
print(f"  TNG300:   {tng_same_host.mean():.1%}  ({tng_same_host.sum():,} / {len(tng_same_host):,})")

In [ ]:
for label, flam_mask, tng_mask in [
    ("Same-host",      pairs["have_same_host"],  tng_same_host),
    ("Different-host", ~pairs["have_same_host"], ~tng_same_host),
]:
    print(f"\n{label}: FLAMINGO N={flam_mask.sum():,}, TNG N={tng_mask.sum():,}")

> Historical interpretation from before the corrected comparison rerun. Reassess quantitative and causal claims using the new outputs.

The same-host and the different-host fractions are different for FLAMINGO and TNG300. Now we will investigate the plots after seperating by same-host and different-host.


In [ ]:
FLAMINGO_COLOR = "#E07B3F"
TNG_COLOR      = "#4C72B0"

bins_sep   = np.linspace(500,  1000, 21)
bins_vr    = np.linspace(-400, 0,   21)
bins_vt    = np.linspace(0,    300, 21)
mass_min = min(np.min(pairs["log_tot_virial_mass"]), np.min(tng_log_m200c))
mass_max = max(np.max(pairs["log_tot_virial_mass"]), np.max(tng_log_m200c))
bins_m200c = np.linspace(mass_min - 1e-6, mass_max + 1e-6, 21)

subpops = [
    ("Same-host",      pairs["have_same_host"],  tng_same_host),
    ("Different-host", ~pairs["have_same_host"], ~tng_same_host),
]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle("FLAMINGO L1_m8 vs TNG300-1 — by host membership", fontsize=14)

panel_specs = [
    ("separation_kpc",      tng_sep,       bins_sep,   "Pair Separation [kpc]",           "Pair Separation"),
    ("vr",                  tng_vr,        bins_vr,    r"Peculiar radial velocity $v_r$ [km/s]",   "Radial Velocity"),
    ("vt",                  tng_vt,        bins_vt,    r"Tangential Velocity $v_t$ [km/s]","Tangential Velocity"),
    ("log_tot_virial_mass", tng_log_m200c, bins_m200c, r"$\log_{10}(M_{200c}/M_\odot)$",  "Host Halo Mass"),
]

for row, (pop_label, flam_mask, tng_mask) in enumerate(subpops):
    for col, (flam_key, tng_arr, bins, xlabel, title) in enumerate(panel_specs):
        ax = axes[row, col]
        
        flam_arr = pairs[flam_key][flam_mask]
        tng_arr_sub = tng_arr[tng_mask]
        
        vf = np.isfinite(flam_arr)
        vt = np.isfinite(tng_arr_sub)
        
        ax.hist(flam_arr[vf], bins=bins, density=True, alpha=0.6,
                color=FLAMINGO_COLOR, label=f"FLAMINGO (N={vf.sum():,})")
        ax.hist(tng_arr_sub[vt], bins=bins, density=True, alpha=0.6,
                color=TNG_COLOR, label=f"TNG300-1 (N={vt.sum():,})")
        
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Density")
        ax.set_title(f"{title}\n({pop_label})")
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(PLOT_DIR / "flamingo_vs_tng_by_host.pdf", dpi=150, bbox_inches="tight")
plt.show()

> Historical interpretation from before the corrected comparison rerun. Reassess quantitative and causal claims using the new outputs.

The plots overlap pretty nicely now. Different-host may be because of the limited sample size but we investigate using Anderson-Darling and bootstrap confidence intervals on different-host where sample size is small.


In [ ]:
# Statistical comparison of FLAMINGO vs TNG300 Local Group analogue pair distributions.

# Distances are descriptive; pairs sharing hosts are not independent observations.
# Histogram bands resample connected-host components within the selected catalogue.
import matplotlib.gridspec as gridspec
from scipy.stats import ks_2samp, energy_distance

FLAMINGO_COLOR = "#E07B3F"
TNG_COLOR      = "#4C72B0"
N_BOOTSTRAP    = 2000
CI             = 95 
ALPHA          = (100 - CI) / 2


def bootstrap_hist_ci(data, bins, groups, n_boot=N_BOOTSTRAP, ci=CI):
    data = np.asarray(data)
    if len(data) != len(groups):
        raise ValueError("Histogram values and host groups are not aligned.")
    blocks = group_blocks(groups)
    rng = np.random.default_rng(42)
    counts = np.empty((n_boot, len(bins) - 1))
    for b in range(n_boot):
        draw = data[resample_blocks(blocks, rng)]
        counts[b], _ = np.histogram(draw, bins=bins, density=True)
    alpha = (100 - ci) / 2
    return np.percentile(counts, alpha, axis=0), np.percentile(counts, 100-alpha, axis=0)

def summarise_stats(flam_arr, tng_arr, label):
    ks = ks_2samp(flam_arr, tng_arr).statistic
    ed = energy_distance(flam_arr, tng_arr)
    print(f"  {label:25s}  KS D={ks:.4f}  energy_dist={ed:.4f} (descriptive)")

subpops = [
    ("Same-host",      pairs["have_same_host"],  tng_same_host),
    ("Different-host", ~pairs["have_same_host"], ~tng_same_host),
]

bins_sep   = np.linspace(500,  1000, 21)
bins_vr    = np.linspace(-400, 0,   21)
bins_vt    = np.linspace(0,    300, 21)
mass_min = min(np.min(pairs["log_tot_virial_mass"]), np.min(tng_log_m200c))
mass_max = max(np.max(pairs["log_tot_virial_mass"]), np.max(tng_log_m200c))
bins_m200c = np.linspace(mass_min - 1e-6, mass_max + 1e-6, 21)

panel_specs = [
    ("separation_kpc",      tng_sep,       bins_sep,   "Pair Separation [kpc]",            "Separation"),
    ("vr",                  tng_vr,        bins_vr,    r"Peculiar radial velocity $v_r$ [km/s]",    "vr"),
    ("vt",                  tng_vt,        bins_vt,    r"Tangential Velocity $v_t$ [km/s]","vt"),
    ("log_tot_virial_mass", tng_log_m200c, bins_m200c, r"$\log_{10}(M_{200c}/M_\odot)$",  "log M200c"),
]

# Print summary statistics

for pop_label, flam_mask, tng_mask in subpops:
    n_flam = flam_mask.sum()
    n_tng  = tng_mask.sum()
    print(f"\n{pop_label}  (FLAMINGO N={n_flam:,}  TNG N={n_tng:,})")
    if n_tng < 50:
        print("  Small selected TNG population; interpret distribution estimates cautiously")
    for flam_key, tng_arr, _, _, short_label in panel_specs:
        flam_vals = pairs[flam_key][flam_mask]
        tng_vals  = tng_arr[tng_mask]
        # Drop non-finite
        flam_vals = flam_vals[np.isfinite(flam_vals)]
        tng_vals  = tng_vals[np.isfinite(tng_vals)]
        summarise_stats(flam_vals, tng_vals, short_label)

# 2x4 plot with bootstrap CIs on TNG histograms

fig = plt.figure(figsize=(22, 10))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

for row, (pop_label, flam_mask, tng_mask) in enumerate(subpops):
    n_flam = flam_mask.sum()
    n_tng  = tng_mask.sum()
    use_bootstrap = n_tng < 1000  # bootstrap CI for small TNG samples

    for col, (flam_key, tng_arr, bins, xlabel, short_label) in enumerate(panel_specs):
        ax = fig.add_subplot(gs[row, col])

        flam_vals = pairs[flam_key][flam_mask]
        tng_vals  = tng_arr[tng_mask]

        vf = np.isfinite(flam_vals)
        vt = np.isfinite(tng_vals)
        flam_vals = flam_vals[vf]
        tng_vals  = tng_vals[vt]

        bin_centres = 0.5 * (bins[:-1] + bins[1:])
        bin_width   = bins[1] - bins[0]

        # FLAMINGO histogram
        ax.hist(flam_vals, bins=bins, density=True, alpha=0.55,
                color=FLAMINGO_COLOR, label=f"FLAMINGO (N={len(flam_vals):,})")

        # TNG histogram
        tng_counts, _ = np.histogram(tng_vals, bins=bins, density=True)
        ax.bar(bin_centres, tng_counts, width=bin_width, alpha=0.55,
               color=TNG_COLOR, label=f"TNG300-1 (N={len(tng_vals):,})")

        # Bootstrap CI on TNG
        if use_bootstrap and len(tng_vals) >= 10:
            lo, hi = bootstrap_hist_ci(tng_vals, bins, tng_groups[tng_mask][vt])
            ax.fill_between(bin_centres, lo, hi,
                            color=TNG_COLOR, alpha=0.25, label=f"{CI}% interval (host bootstrap)")

        # Descriptive distances, without independent-row significance stars.
        if len(tng_vals) >= 10:
            stat = ks_2samp(flam_vals, tng_vals).statistic
            ed = energy_distance(flam_vals, tng_vals)
            annotation = f"KS D={stat:.2f}\nED={ed:.3f}"
            ax.text(0.97, 0.97, annotation,
                    transform=ax.transAxes, fontsize=7.5,
                    ha="right", va="top",
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.7, ec="gray"))

        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel("Density", fontsize=9)
        ax.set_title(f"{short_label}\n({pop_label})", fontsize=9.5)
        ax.legend(fontsize=7.5)
        ax.tick_params(labelsize=8)

fig.suptitle(
    "FLAMINGO L1_m8 vs TNG300-1 — Local Group Analogues by host membership\n"
    "Shaded bands = 95% connected-host bootstrap interval, conditional on selection (N<1000)",
    fontsize=12, y=1.01
)

plt.savefig(PLOT_DIR / "flamingo_vs_tng_stats.pdf", dpi=150, bbox_inches="tight")
plt.show()

### Summary Statistics


In [ ]:
import pandas as pd

quantities = {
    "separation":     (pairs["separation_kpc"],      tng_sep),
    "vr":             (pairs["vr"],                  tng_vr),
    "vt":             (pairs["vt"],                  tng_vt),
    "log_m200c":      (pairs["log_tot_virial_mass"],  tng_log_m200c),
    "log_mass_ratio": (pairs["log_mass_ratio"],       tng_log_mass_ratio),
}

rows = []
for name, (flam_arr, tng_arr) in quantities.items():
    for sim, arr in [("FLAMINGO", flam_arr), ("TNG300", tng_arr)]:
        a = arr[np.isfinite(arr)]
        rows.append({
            "quantity":  name,
            "sim":       sim,
            "N":         len(a),
            "mean":      np.mean(a),
            "median":    np.median(a),
            "std":       np.std(a),
            "IQR":       np.percentile(a, 75) - np.percentile(a, 25),
            "p5":        np.percentile(a, 5),
            "p95":       np.percentile(a, 95),
        })

summary_df = pd.DataFrame(rows)
pd.set_option("display.float_format", "{:.3f}".format)
display(summary_df)

### Cumulative Distribution Functions


In [ ]:
# ECDFs of all current pairs and each host population, including the mass ratio.
cdf_specs = [
    ("separation_kpc", tng_sep, "Separation [kpc]"),
    ("vr", tng_vr, r"Peculiar $v_r$ [km/s]"),
    ("vt", tng_vt, r"$v_t$ [km/s]"),
    ("log_tot_virial_mass", tng_log_m200c, r"$\log_{10}(M_{200c}/M_\odot)$"),
    ("log_mass_ratio", tng_log_mass_ratio, r"$\log_{10}(M_{\rm smaller}/M_{\rm larger})$"),
]
cdf_populations = [
    ("All pairs", np.ones(len(pairs["i"]), dtype=bool), np.ones(len(tng_sep), dtype=bool)),
    ("Same host", pairs["have_same_host"], tng_same_host),
    ("Different hosts", ~pairs["have_same_host"], ~tng_same_host),
]
fig, axes = plt.subplots(3, 5, figsize=(23, 13))
for row, (population, fm, tm) in enumerate(cdf_populations):
    for ax, (key, tng_values, xlabel) in zip(axes[row], cdf_specs):
        for values, color, label in [
            (pairs[key][fm], FLAMINGO_COLOR, "FLAMINGO"),
            (tng_values[tm], TNG_COLOR, "TNG300"),
        ]:
            values = np.sort(values[np.isfinite(values)])
            ax.step(values, np.arange(1, len(values)+1)/len(values), where="post",
                    color=color, label=f"{label} (N={len(values):,})")
        ax.set(xlabel=xlabel, ylabel="ECDF", title=population)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
fig.suptitle("FLAMINGO vs TNG300 pair distributions")
plt.tight_layout()
plt.savefig(PLOT_DIR / "pair_ecdfs_by_host.pdf", bbox_inches="tight")
plt.show()

> Historical interpretation from before the corrected comparison rerun. Reassess quantitative and causal claims using the new outputs.

### KS and Wasserstein distances


In [ ]:
from scipy.stats import ks_2samp, wasserstein_distance

print(f"{'Quantity':<20}  {'KS stat':>8}  {'Wasserstein':>12}")

for name, (flam_arr, tng_arr) in quantities.items():
    f = flam_arr[np.isfinite(flam_arr)]
    t = tng_arr[np.isfinite(tng_arr)]
    ks,  _ = ks_2samp(f, t)
    wd     = wasserstein_distance(f, t)
    print(f"{name:<20}  {ks:>8.4f}  {wd:>12.4f}")

In [ ]:
# Normalizing everything to make comparison easier between different units
quantities_norm = {
    "separation":     (pairs["separation_kpc"],      tng_sep),
    "vr":             (pairs["vr"],                  tng_vr),
    "vt":             (pairs["vt"],                  tng_vt),
    "log_m200c":      (pairs["log_tot_virial_mass"],  tng_log_m200c),
    "log_mass_ratio": (pairs["log_mass_ratio"],       tng_log_mass_ratio),
}

print(f"{'Quantity':<20}  {'KS stat':>8}  {'Wasserstein':>12}  {'W (normalised)':>15}  {'pooled std':>10}")

for name, (flam_arr, tng_arr) in quantities_norm.items():
    f = flam_arr[np.isfinite(flam_arr)]
    t = tng_arr[np.isfinite(tng_arr)]
    
    ks,  _ = ks_2samp(f, t)
    wd     = wasserstein_distance(f, t)
    
    # pooled std
    pooled_std = np.sqrt(
        (len(f) * np.std(f)**2 + len(t) * np.std(t)**2) / (len(f) + len(t))
    )
    wd_norm = wd / pooled_std

    print(f"{name:<20}  {ks:>8.4f}  {wd:>12.4f}  {wd_norm:>15.4f}  {pooled_std:>10.4f}")

> Historical interpretation from before the corrected comparison rerun. Reassess quantitative and causal claims using the new outputs.

### Revised interpretation pending corrected plots

The TNG300 box length is 205 / 0.6774 = 302.628 Mpc at z=0, and its volume is
27,715,709.856 Mpc³. The FLAMINGO/TNG volume ratio is 36.081. Pair-density
comparisons must use the corrected pair count in the v3 selection manifest.
Earlier medians, counts, and the inferred relative-density explanation are historical.
Selection definitions still differ in catalogue finder, mass resolution and quality flags;
do not attribute every distribution difference to feedback physics.


In [ ]:
# Corrected volumes and selected-pair density, using the v3 reconstruction manifest.
import json
import pandas as pd
selection = json.loads((tng_dir / "selection_manifest.json").read_text())
volume_tng = tng_volume_mpc3({"BoxSize": selection["BoxSize_ckpc_h"],
                             "HubbleParam": selection["h"], "Redshift": 0})
density_summary = pd.DataFrame({"simulation": ["FLAMINGO", "TNG300"],
    "pairs": [len(pairs["i"]), selection["pair_count"]],
    "volume_Mpc3": [1000.0**3, volume_tng]})
density_summary["pairs_per_Mpc3"] = density_summary["pairs"] / density_summary["volume_Mpc3"]
print(density_summary.to_string(index=False))
print("FLAMINGO / TNG pair-density ratio:", density_summary.pairs_per_Mpc3.iloc[0] / density_summary.pairs_per_Mpc3.iloc[1])
density_summary.to_csv(PLOT_DIR / "pair_density.csv", index=False)